# 3.1 趋势类指标

## 学习目标
- 理解移动均线（SMA / EMA）的构造原理
- 掌握 MACD 的计算与信号解读
- 用均线金叉死叉构造交易信号

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams['figure.figsize'] = (13, 5)

data = yf.download('AAPL', start='2022-01-01', end='2024-01-01', progress=False)
close = data['Close'].squeeze()
print('数据加载完成 ✅')

## 1. SMA — 简单移动均线

$$SMA_n(t) = \frac{1}{n} \sum_{i=0}^{n-1} P_{t-i}$$

- 平滑价格噪声，识别趋势方向
- 缺点：**滞后**，对近期价格无特殊加权

In [ ]:
sma20 = close.rolling(20).mean()
sma60 = close.rolling(60).mean()

# 金叉/死叉信号
cross_up = (sma20 > sma60) & (sma20.shift(1) <= sma60.shift(1))   # 金叉
cross_dn = (sma20 < sma60) & (sma20.shift(1) >= sma60.shift(1))   # 死叉

fig, ax = plt.subplots()
ax.plot(close.index, close.values, label='收盘价', linewidth=1, alpha=0.7)
ax.plot(sma20.index, sma20.values, label='SMA20', linewidth=1.5, color='orange')
ax.plot(sma60.index, sma60.values, label='SMA60', linewidth=1.5, color='red')
ax.scatter(close.index[cross_up], close.values[cross_up], marker='^', color='green', s=120, zorder=5, label='金叉')
ax.scatter(close.index[cross_dn], close.values[cross_dn], marker='v', color='red', s=120, zorder=5, label='死叉')
ax.set_title('SMA20 / SMA60 金叉死叉策略', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. EMA — 指数移动均线

$$EMA_t = \alpha \cdot P_t + (1-\alpha) \cdot EMA_{t-1}, \quad \alpha = \frac{2}{n+1}$$

- 对**近期价格**赋予更高权重，反应更灵敏

In [ ]:
ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()

fig, ax = plt.subplots()
ax.plot(close.index, close.values, label='Close', linewidth=1, alpha=0.5)
ax.plot(sma20.index, sma20.values, label='SMA20（慢）', linewidth=1.5, linestyle='--')
ax.plot(ema12.index, ema12.values, label='EMA12（快）', linewidth=1.5)
ax.set_title('SMA vs EMA 对比（SMA 更滞后）', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. MACD — 指数平滑异同移动平均线

$$MACD = EMA_{12} - EMA_{26}$$
$$Signal = EMA_9(MACD)$$
$$Histogram = MACD - Signal$$

In [ ]:
macd_line = ema12 - ema26
signal_line = macd_line.ewm(span=9, adjust=False).mean()
histogram = macd_line - signal_line

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# 价格图
ax1.plot(close.index, close.values, linewidth=1.2, color='steelblue')
ax1.set_title('AAPL 收盘价', fontsize=12)
ax1.set_ylabel('价格')
ax1.grid(alpha=0.3)

# MACD 图
ax2.plot(macd_line.index, macd_line.values, label='MACD', linewidth=1.5, color='blue')
ax2.plot(signal_line.index, signal_line.values, label='Signal', linewidth=1.5, color='orange')
colors = ['green' if v >= 0 else 'red' for v in histogram.values]
ax2.bar(histogram.index, histogram.values, color=colors, alpha=0.4, label='Histogram')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('MACD 指标', fontsize=12)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('📌 交易信号：MACD 线从下穿越 Signal 线 → 买入信号')

## 🎯 练习

1. 将 SMA 参数从 (20, 60) 改为 (5, 20)，观察金叉/死叉频率变化。
2. MACD 的直方图（Histogram）在什么情况下达到峰值？这意味着什么？
3. 实现一个简单规则：MACD 线上穿 Signal 线买入，下穿卖出，计算年化收益。

---
**下一节** → `02_momentum_indicators.ipynb`